# b0 · Historical-volatility benchmark (`hist_rw`)

Random-walk forecast of monthly realised variance for the five cyclical ETFs. The forecast made at the end of month *t* for month *t+1* is month *t*'s realised variance, `RV_hat(t+1) = RV(t)`. It has no estimated parameters. In a volatility-managed portfolio it gives the weight `sigma2_bar / RV(t)`, the specification of Moreira & Muir (2017).

Output: `forecasts/hist_rw_forecasts.csv`. It is the benchmark in `evaluation/e6_naive_vs_all_models.ipynb`.

In [1]:
import sys
from pathlib import Path

_checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models" / "research_config.py").is_file()),
    None,
)
if _checkout is None:
    raise FileNotFoundError("Start the notebook kernel from within this project checkout.")
sys.path.insert(0, str(_checkout / "models"))
from utils.paths import PROJECT_ROOT, MODELS_DIR, DATASET_DIR, FORECAST_DIR, FORECAST_RESULTS_DIR

import os
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)


RV_PATH = os.path.join(PROJECT_ROOT, "dataset", "price", "clean", "realized_variance.csv")
FORECASTS_DIR = os.path.join(PROJECT_ROOT, "models", "forecasting", "forecasts")

CYCLICAL_TICKERS = ["XLI", "XLB", "XLE", "XLY", "XLF"]
MODEL = "hist_rw"
# Same forecast origins as the other forecast files: 2008-02 (for holding month 2008-03) onwards
FIRST_ORIGIN = pd.Timestamp("2008-02-29")

## Data

`rv_1m` from the shared `realized_variance.csv`, the same panel the OLS and Elastic Net models use. Each ETF's months must be consecutive with positive RV.

In [2]:
rv = pd.read_csv(RV_PATH, parse_dates=["date"])

rv_series = {}
for etf, g in rv.sort_values("date").groupby("etf"):
    s = g.set_index("date")["rv_1m"]
    periods = pd.PeriodIndex(s.index, freq="M")
    assert np.all(np.diff(periods.asi8) == 1), f"{etf}: months are not consecutive"
    assert s.notna().all() and (s > 0).all(), f"{etf}: missing or non-positive RV"
    rv_series[etf] = s
    print(f"{etf}: {len(s)} rows, {s.index.min().date()} to {s.index.max().date()}")

XLB: 275 rows, 2003-02-28 to 2025-12-31
XLE: 275 rows, 2003-02-28 to 2025-12-31
XLF: 275 rows, 2003-02-28 to 2025-12-31
XLI: 275 rows, 2003-02-28 to 2025-12-31
XLY: 275 rows, 2003-02-28 to 2025-12-31


## Forecasts

A row dated *t* (forecast origin) holds the forecast for `target_date` *t+1*.

- `fc_log = log RV(t)`, `fc_level = RV(t)`.
- `sigma2` is NaN: the forecast is already in levels, so there is no log-normal correction.

In [3]:
def build_hist_rw(s):
    # random-walk forecasts for one ETF's monthly RV series
    rv_vals = s.to_numpy()
    return pd.DataFrame({
        "date": s.index[:-1],            # forecast origin t
        "target_date": s.index[1:],      # month being forecast, t+1
        "actual_logRV": np.log(rv_vals[1:]),
        "actual_RV": rv_vals[1:],
        "fc_log": np.log(rv_vals[:-1]),
        "sigma2": np.nan,
        "fc_level": rv_vals[:-1],
    })

## Run

Fast, so the forecasts are always rebuilt and saved.

In [4]:
SAVED_PATH = os.path.join(FORECASTS_DIR, "hist_rw_forecasts.csv")

frames = []
for etf, s in rv_series.items():
    tidy = build_hist_rw(s)
    tidy.insert(0, "etf", etf)
    tidy.insert(0, "model", MODEL)
    frames.append(tidy)
tidy_df = pd.concat(frames, ignore_index=True)
tidy_df = tidy_df[tidy_df["date"] >= FIRST_ORIGIN].reset_index(drop=True)

os.makedirs(FORECASTS_DIR, exist_ok=True)
tidy_df.to_csv(SAVED_PATH, index=False, date_format="%Y-%m-%d")
print(tidy_df.groupby("etf")["date"].agg(["min", "max", "count"]))
print(f"Saved {len(tidy_df)} rows to {SAVED_PATH}")

           min        max  count
etf                             
XLB 2008-02-29 2025-11-30    214
XLE 2008-02-29 2025-11-30    214
XLF 2008-02-29 2025-11-30    214
XLI 2008-02-29 2025-11-30    214
XLY 2008-02-29 2025-11-30    214
Saved 1070 rows to ../../..\models\forecasting\forecasts\hist_rw_forecasts.csv


## Accuracy

Per-ticker and pooled MSE on log RV and QLIKE on the level forecast.

In [5]:
def qlike(actual_rv, fc_level):
    ratio = actual_rv / fc_level
    return ratio - np.log(ratio) - 1


d = tidy_df.assign(sq_err_log=(tidy_df["actual_logRV"] - tidy_df["fc_log"]) ** 2,
                   qlike=qlike(tidy_df["actual_RV"], tidy_df["fc_level"]))
agg = dict(MSE_log=("sq_err_log", "mean"), QLIKE=("qlike", "mean"))
per_ticker = d.groupby("etf").agg(**agg)
pooled = d[["sq_err_log", "qlike"]].mean().set_axis(["MSE_log", "QLIKE"]).rename("pooled")
acc = pd.concat([per_ticker, pooled.to_frame().T]).reindex([t for t in CYCLICAL_TICKERS if t in per_ticker.index] + ["pooled"])
print(acc.round(4))

        MSE_log   QLIKE
XLI      0.6302  0.4200
XLB      0.5409  0.3589
XLE      0.5116  0.3480
XLY      0.5758  0.3775
XLF      0.7564  0.5268
pooled   0.6030  0.4063
